# AudioRouter quick reproduction

This notebook verifies the released checkpoints, runs a one-video inference demo, and exposes the exact commands used for the three reported validation suites. The full evaluations are opt-in because they take several GPU-hours.

AudioRouter uses audio only to route native visual tokens; audio embeddings never enter the language model.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'AudioRouter').is_dir(), f'Run this notebook from the repository or notebooks directory: {ROOT}'

for path in (ROOT, ROOT / 'LLaVA-NeXT', ROOT / 'lmms-eval'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

GPU = '0'
BEATS_CHECKPOINT = Path(os.getenv(
    'BEATS_CHECKPOINT',
    '/nvme_data/pkt/huggingface/modules/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt1.pt',
))
env = os.environ.copy()
env.update({
    'CUDA_VISIBLE_DEVICES': GPU,
    'PYTHONNOUSERSITE': '1',
    'PYTHONPATH': os.pathsep.join(map(str, (ROOT / 'lmms-eval', ROOT / 'LLaVA-NeXT', ROOT))),
    'HF_HUB_OFFLINE': env.get('HF_HUB_OFFLINE', '1'),
    'TRANSFORMERS_OFFLINE': env.get('TRANSFORMERS_OFFLINE', '1'),
    'TOKENIZERS_PARALLELISM': 'false',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'BEATS_CHECKPOINT': str(BEATS_CHECKPOINT),
    'BEATS_EMBEDDING_CACHE': str(ROOT / 'results' / 'cache' / 'beats'),
})
print('repository:', ROOT)
print('Python:', sys.executable)
print('GPU:', GPU)
print('BEATs:', BEATS_CHECKPOINT, 'exists=' + str(BEATS_CHECKPOINT.is_file()))

## 1. Verify checkpoint files

A hash mismatch means the result is not directly comparable with the table in the README.

In [ ]:
EXPECTED_SHA256 = {
    'videomme_adbt_epoch3.pt': '1230ce639c8621ffcbb9d3d906179a454a1015475f0ee1b7d3957dc50ce631ec',
    'mlvu_ego_adbt_epoch3.pt': '53835943f7d21129520ac532c05c169f422a40b617bb39949dddbae6e4313a1d',
    'mlvu_full_adbt_epoch2.pt': '043b2c7227d841197244816db3c8032f671e4d7e8671b20617f3a41c721cb999',
    'streamingbench_realtime_adbt_epoch3.pt': '9d12f5c70c7b1bcdea984a324765dc31de223aca38154c7301328f440fbdb949',
}

def sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

for name, expected in EXPECTED_SHA256.items():
    path = ROOT / 'ckpt' / name
    actual = sha256(path) if path.is_file() else 'MISSING'
    print(f'{name:45s}', 'OK' if actual == expected else f'FAIL ({actual})')

## 2. One-video inference and visualization

Set a local video and its four answer options. This is the fastest end-to-end check: it loads the frozen VLM and BEATs, performs real AudioRouter inference, and produces an annotated MP4 plus a JSON sidecar. Use `QUERY_TIME=None` for whole-video QA or a positive timestamp for causal real-time QA.

In [ ]:
VIDEO = Path('/path/to/example.mp4')
QUESTION = 'What happens after the person opens the door?'
OPTIONS = ['They sit down', 'They leave', 'They wave', 'They cook']
QUERY_TIME = None
RUN_DEMO = False  # Set to True after editing VIDEO/question/options.

demo_output = ROOT / 'results' / 'demo' / 'audiorouter_inference.mp4'
demo_command = [
    sys.executable, str(ROOT / 'scripts' / 'render_inference_demo.py'),
    '--video', str(VIDEO),
    '--question', QUESTION,
    '--options', *OPTIONS,
    '--checkpoint', str(ROOT / 'ckpt' / 'videomme_adbt_epoch3.pt'),
    '--attention-temperature', '0.03',
    '--beats-checkpoint', str(BEATS_CHECKPOINT),
    '--output', str(demo_output),
    '--include-audio',
]
if QUERY_TIME is not None:
    demo_command += ['--query-time', str(QUERY_TIME)]
print(' '.join(map(str, demo_command)))
if RUN_DEMO:
    if not VIDEO.is_file():
        raise FileNotFoundError(VIDEO)
    subprocess.run(demo_command, cwd=ROOT, env=env, check=True)

## 3. Full held-out validation

These commands use `fps=auto`, no frame cap, real audio, and the released held-out splits. Set only the suite you want to run to `True`. VideoMME writes lmms-eval result files; MLVU and StreamingBench write one self-contained JSON each. Media paths in normalized manifests must exist on your machine.

In [ ]:
RUN_VIDEOMME = False
RUN_MLVU_FULL = False
RUN_STREAMINGBENCH = False

videomme_commands = [
    ['bash', 'scripts/run_phase5_short_ablation.sh', '0', 'real', 'ckpt/videomme_adbt_epoch3.pt', 'results/verify-videomme-short', '0.03', 'videomme_short'],
    ['bash', 'scripts/run_phase5_short_ablation.sh', '0', 'real', 'ckpt/videomme_adbt_epoch3.pt', 'results/verify-videomme-medium', '0.03', 'videomme_medium'],
    ['bash', 'scripts/run_phase5_short_ablation.sh', '0', 'real', 'ckpt/videomme_adbt_epoch3.pt', 'results/verify-videomme-long', '0.03', 'videomme_long'],
]
mlvu_command = [
    sys.executable, '-m', 'AudioRouter.eval_adbt_mcqa',
    '--dataset-manifest', 'results/dataset_manifests/mlvu_full_mcqa.json',
    '--split-file', 'results/dataset_splits/mlvu_full_mcqa_seed1234_pathgroup_80_20.json',
    '--checkpoint', 'ckpt/mlvu_full_adbt_epoch2.pt',
    '--output', 'results/verify-mlvu-full.json',
    '--fps', 'auto', '--max-frames', '0', '--audio-ablation', 'real',
    '--attention-temperature', '0.015',
    '--vision-batch-size', '8', '--beats-batch-size', '8',
    '--beats-checkpoint', str(BEATS_CHECKPOINT),
]
streamingbench_command = [
    sys.executable, '-m', 'AudioRouter.eval_adbt_mcqa',
    '--dataset-manifest', 'results/dataset_manifests/streamingbench_realtime.json',
    '--split-file', 'results/dataset_splits/streamingbench_realtime_seed1234_80_20.json',
    '--checkpoint', 'ckpt/streamingbench_realtime_adbt_epoch3.pt',
    '--output', 'results/verify-streamingbench-realtime.json',
    '--fps', 'auto', '--max-frames', '0', '--audio-ablation', 'real',
    '--attention-temperature', '0.05',
    '--vision-batch-size', '8', '--beats-batch-size', '8',
    '--beats-checkpoint', str(BEATS_CHECKPOINT),
]

if RUN_VIDEOMME:
    for command in videomme_commands:
        subprocess.run(command, cwd=ROOT, env=env, check=True)
if RUN_MLVU_FULL:
    subprocess.run(mlvu_command, cwd=ROOT, env=env, check=True)
if RUN_STREAMINGBENCH:
    subprocess.run(streamingbench_command, cwd=ROOT, env=env, check=True)

## 4. Inspect MCQA results

Expected held-out results: MLVU Full **291/429 (67.8322%)** and StreamingBench Real-Time **362/500 (72.40%)**.

In [ ]:
for relative in ('results/verify-mlvu-full.json', 'results/verify-streamingbench-realtime.json'):
    path = ROOT / relative
    if not path.is_file():
        print(relative, 'not run yet')
        continue
    result = json.loads(path.read_text())
    overall = result['overall']
    print(
        f"{result['dataset']}: {overall['option_correct']}/{overall['examples']} "
        f"= {100 * overall['option_accuracy']:.4f}% | "
        f"frames={result['sampled_frames']:,} | tau={result['attention_temperature']}"
    )